# ClassifAI Demo

This demo uses a mock occupations dataset to show how ClassifAI matches unlabelled job descriptions to SOC codes using an existing knowledgebase.

In [ ]:
import glob
import os
import string
import pandas as pd
from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import (
    HookBase,
    DeduplicationHook,
    CapitalisationStandardisingHook,
)
from classifai.vectorisers import FastEmbedVectoriser

### Setup

The cells below load the data and build the vector store.

In [18]:
# Prepare knowledgebase - VectorStore expects 'label' and 'text' columns
coded_df = pd.read_csv("../data/mock_soc_dataset.csv")
coded_df["label"] = coded_df["soc_code"]
coded_df["text"] = coded_df["role"] + ": " + coded_df["description"]
coded_df.to_csv("../data/mock_vector_store_data.csv", index=False)

# Prepare queries - VectorStore search expects 'id' and 'query' columns
uncoded_df = pd.read_csv("../data/mock_uncoded_soc_responses.csv")
uncoded_df["query"] = uncoded_df["role"] + ": " + uncoded_df["description"]
uncoded_df["id"] = uncoded_df.index
input_data = VectorStoreSearchInput.from_data(uncoded_df)

# We use a local model as mybinder cant download reliably from huggingface or other model repositories.
# Most applications of ClassifAI would just specify a model_name and it will automatically download the files.
search_pattern = "../data/fastembed_cache/models--*--*/snapshots/*"
matching_dirs = glob.glob(search_pattern)

if matching_dirs:
    model_path = matching_dirs[0]
else:
    raise RuntimeError(
        "The pre-cached FastEmbed model directory could not be located. "
        "Please ensure the environment's build step (postBuild) completed successfully "
        "or locate/download the model weights manually. If this issue persists in the "
        "live demo environment, please report it to the ClassifAI team via GitHub."
    )

# Build vector store
vectoriser = FastEmbedVectoriser(
    model_name="BAAI/bge-small-en-v1.5",
    specific_model_path=model_path,
)

soc_vector_store = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    meta_data={"role": str},
    hooks={},
)


INFO - Processing file: ../data/mock_vector_store_data.csv...

100%|██████████| 1/1 [00:00<00:00, 42.43it/s]


---
## 1. Basic Search

Pass a batch of unlabelled job descriptions and get back the best matching SOC code for each one.

In [19]:
soc_vector_store.search(query=input_data, n_results=1)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 48.59it/s]


,query_id,query_text,doc_label,doc_text,rank,score,role
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Vegetable farmer
1,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646,Dairy farmer
2,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518,Software developer


Set `n_results` higher to return a ranked shortlist rather than a single match — useful when you want to surface alternatives for review.

In [20]:
soc_vector_store.search(query=input_data, n_results=3)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 53.15it/s]


,query_id,query_text,doc_label,doc_text,rank,score,role
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Vegetable farmer
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Fruit farmer: Grows and harvests fruits such a...,2,0.820159,Fruit farmer
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,3,0.782137,Dairy farmer
3,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646,Dairy farmer
4,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Vegetable farmer: Cultivates and harvests vege...,2,0.738995,Vegetable farmer
5,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Fruit farmer: Grows and harvests fruits such a...,3,0.737185,Fruit farmer
6,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518,Software developer
7,2,Machine Learning Engineer: Designs and deploys...,107,Web developer: Builds and maintains websites a...,2,0.681587,Web developer
8,2,Machine Learning Engineer: Designs and deploys...,104,"Carpenter: Constructs, installs, and repairs w...",3,0.641518,Carpenter


---
## 2. Metadata in Results

The `meta_data` parameter controls which extra knowledgebase columns are returned alongside each match. In this example we added the "role" column to the `meta_data` which is why we see it in the search result.

```diff
  soc_vector_store = VectorStore(
      file_name="../data/mock_vector_store_data.csv",
      data_type="csv",
      vectoriser=vectoriser,
+     meta_data={"role": str},
  )
```

In [21]:
soc_vector_store.search(query=input_data, n_results=1)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 55.54it/s]


,query_id,query_text,doc_label,doc_text,rank,score,role
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Vegetable farmer
1,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646,Dairy farmer
2,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518,Software developer


---
## 3. Hooks

Hooks are pre or post processing functions attached to the vector store - They run automatically on every search call.
We provide some basic ones, but you can make your own!
> This helps keep transformation logic organised and repeatable.

### Capitalisation Standardising Hook

Normalises query casing before embedding. Attach it as to the hook dictionary and it will run on every query automatically.
> The key to the dictionary can be called whatever you want, this allows you to group similar hooks

In [22]:
soc_vs_lower = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={
        "search_preprocess": CapitalisationStandardisingHook(
            method="lower", colname="query"
        )
    },
)

# Deliberately inconsistent casing — the hook normalises before embedding
messy_input = VectorStoreSearchInput({
    "id": [0, 1, 2],
    "query": ["TOMATO FARMER", "Machine Learning ENGINEER", "pHd StUdEnT"],
})

soc_vs_lower.search(query=messy_input, n_results=1)

INFO - Processing file: ../data/mock_vector_store_data.csv...

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 256.08it/s]


,query_id,query_text,doc_label,doc_text,rank,score
0,0,tomato farmer,101,Vegetable farmer: Cultivates and harvests vege...,1,0.769499
1,1,machine learning engineer,107,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,phd student,107,"Software developer: Designs, writes, and tests...",1,0.627140


### Deduplication Hook

With `n_results > 1`, the same doc_label can appear multiple times if several knowledgebase entries share that label. `DeduplicationHook` collapses duplicates to one result per label, keeping only the highest-scoring match.

In [23]:
# Without deduplication — the same SOC code can appear multiple times per query
soc_result = soc_vector_store.search(query=input_data, n_results=5)

soc_result

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 38.28it/s]


,query_id,query_text,doc_label,doc_text,rank,score,role
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Vegetable farmer
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Fruit farmer: Grows and harvests fruits such a...,2,0.820159,Fruit farmer
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,3,0.782137,Dairy farmer
3,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,"Sheep farmer: Raises sheep for wool, meat, and...",4,0.721795,Sheep farmer
4,0,Tomato Farmer: Cultivates and harvests tomatoe...,103,Construction laborer: Performs physical tasks ...,5,0.632407,Construction laborer
5,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646,Dairy farmer
6,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Vegetable farmer: Cultivates and harvests vege...,2,0.738995,Vegetable farmer
7,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Fruit farmer: Grows and harvests fruits such a...,3,0.737185,Fruit farmer
8,1,Cow Farmer: Manages dairy and beef cattle oper...,102,"Sheep farmer: Raises sheep for wool, meat, and...",4,0.715102,Sheep farmer
9,1,Cow Farmer: Manages dairy and beef cattle oper...,103,Construction laborer: Performs physical tasks ...,5,0.642462,Construction laborer


In [26]:
soc_vs_dedup = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={"search_postprocess": DeduplicationHook(score_aggregation_method="max")},
)

# Same search — each SOC code now appears at most once per query
soc_vs_dedup.search(query=input_data, n_results=5)

INFO - Processing file: ../data/mock_vector_store_data.csv...

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 91.39it/s]


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,2,0.782137
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,103,Construction laborer: Performs physical tasks ...,3,0.632407
3,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Vegetable farmer: Cultivates and harvests vege...,2,0.738995
4,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646
5,1,Cow Farmer: Manages dairy and beef cattle oper...,103,Construction laborer: Performs physical tasks ...,3,0.642462
6,2,Machine Learning Engineer: Designs and deploys...,102,Dairy farmer: Manages cows for milk production...,3,0.626797
7,2,Machine Learning Engineer: Designs and deploys...,104,"Carpenter: Constructs, installs, and repairs w...",2,0.641518
8,2,Machine Learning Engineer: Designs and deploys...,105,"Electrician: Installs, maintains, and repairs ...",4,0.559389
9,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518


---
## 4. Custom Hooks

Extend `HookBase` to add any preprocessing step you need. The hook receives the search input dataclass and must return the same type.

In [27]:
class RemovePunctuationHook(HookBase):
    def __call__(self, data):
        data["query"] = [
            q.translate(str.maketrans("", "", string.punctuation))
            for q in data["query"]
        ]
        return data


soc_vs_custom = VectorStore(
    file_name="../data/mock_vector_store_data.csv",
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    hooks={"search_preprocess": RemovePunctuationHook()},
)

punctuated_input = VectorStoreSearchInput({
    "id": [0, 1],
    "query": ["Tomato... Farmer!!!", "Machine-Learning Engineer (AI/ML)"],
})


soc_vs_custom.search(query=punctuated_input, n_results=1)

INFO - Processing file: ../data/mock_vector_store_data.csv...

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 233.43it/s]


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer,101,Vegetable farmer: Cultivates and harvests vege...,1,0.769499
1,1,MachineLearning Engineer AIML,107,"Software developer: Designs, writes, and tests...",1,0.664222
